# Bayesian Core Analysis: Regresyon, Panel ve Zaman Yapılı Modeller

Bu notebook, regresyon, panel ve zaman yapılı modelleri AIC ve BIC kriterlerini ön plana çıkararak karşılaştırır.

Modelleme politikası:

- Anlamlı değişkenler önce dahil edilir.
- Anlamlı olmayan değişkenler, korelasyon veya etki büyüklüğü karşılaştırmayı haklı kılacak kadar güçlüyse yine de test edilir.
- Tercih edilen model, yalnızca anlamlılığa değil, AIC/BIC'e dayalı olarak istatistiksel açıdan savunulabilir adaylar arasından seçilir.


In [ ]:
# --- Kütüphaneler ---
from pathlib import Path

import numpy as np
import pandas as pd
import statsmodels.formula.api as smf
from IPython.display import display, Markdown


In [ ]:
# --- Veri yükleme ve Bayesian değişken inşası ---
# Bu blok, 01_bayesian_descriptive_and_normality.ipynb ile birebir aynı
# MAP selection mantığını kullanır. Böylece tüm notebook'larda
# obs_value_bayes_selected_signed_log aynı yöntemle üretilmiş olur.

project_root = Path.cwd()
while not (project_root / 'data').exists() and project_root != project_root.parent:
    project_root = project_root.parent

data_path = project_root / 'data' / 'processed' / 'OECD_FDI_Except_Pointless.csv'
df = pd.read_csv(data_path, sep=';', encoding='latin1', engine='python', on_bad_lines='skip')

# Ham sayısal değer — nokta binlik ayracı olarak kullanılmış, kaldırılıyor
df['OBS_VALUE_NUM'] = pd.to_numeric(
    df['OBS_VALUE'].astype(str).str.replace('.', '', regex=False),
    errors='coerce'
)

# Eksik değer flag'i — eksikliğin kendisi bir bilgi taşıyabilir
df['obs_missing_flag']    = df['OBS_VALUE_NUM'].isna().astype(int)

# Eksik değerleri global medyanla doldur; signed-log dönüşümü uygula
df['obs_value_imputed']   = df['OBS_VALUE_NUM'].fillna(df['OBS_VALUE_NUM'].median())
df['obs_value_signed_log'] = np.sign(df['obs_value_imputed']) * np.log1p(np.abs(df['obs_value_imputed']))

# --- Empirical Bayes shrinkage: grup düzeyinde posterior ortalamalar ---
global_mean = df['obs_value_signed_log'].mean()
global_var  = df['obs_value_signed_log'].var(ddof=1)

def empirical_bayes_mean(data, group_col, value_col='obs_value_signed_log'):
    grouped  = data.groupby(group_col)[value_col]
    stats_df = grouped.agg(['mean', 'count', 'var']).reset_index()
    between_var = max(stats_df['mean'].var(ddof=1), 1e-6)
    within_var  = max(stats_df['var'].fillna(global_var).mean(), 1e-6)
    stats_df['shrinkage_weight'] = (
        (stats_df['count'] * between_var) /
        (stats_df['count'] * between_var + within_var)
    )
    stats_df['posterior_mean'] = (
        stats_df['shrinkage_weight'] * stats_df['mean'] +
        (1 - stats_df['shrinkage_weight']) * global_mean
    )
    return stats_df[[group_col, 'posterior_mean']]

for group_col, new_col in [
    ('REF_AREA',    'bayes_ref_area'),
    ('ACTIVITY',    'bayes_activity'),
    ('TYPE_ENTITY', 'bayes_type_entity'),
]:
    post = empirical_bayes_mean(df, group_col).rename(columns={'posterior_mean': new_col})
    df   = df.merge(post, on=group_col, how='left')

df['bayes_global'] = global_mean

# --- MAP (Maximum A Posteriori) seçimi ---
# Gaussian log-likelihood hesaplanarak dört aday arasından
# gözleme en yakın posterior kaynağı seçilir
sigma2         = max(df['obs_value_signed_log'].var(ddof=1), 1e-6)
candidate_cols = ['bayes_ref_area', 'bayes_activity', 'bayes_type_entity', 'bayes_global']

for col in candidate_cols:
    df[f'loglik_{col}'] = -0.5 * ((df['obs_value_signed_log'] - df[col]) ** 2) / sigma2

# Softmax normalasyonu (log-sum-exp trick ile sayısal kararlılık)
loglik_cols = [f'loglik_{col}' for col in candidate_cols]
max_loglik  = df[loglik_cols].max(axis=1)
exp_shifted = np.exp(df[loglik_cols].sub(max_loglik, axis=0))
posterior_probs = exp_shifted.div(exp_shifted.sum(axis=1), axis=0)
posterior_probs.columns = [c.replace('loglik_', 'posterior_prob_') for c in posterior_probs.columns]
df = pd.concat([df, posterior_probs], axis=1)

df['bayes_best_source']      = (
    posterior_probs.idxmax(axis=1)
    .str.replace('posterior_prob_', '', regex=False)
)
df['bayes_best_probability'] = posterior_probs.max(axis=1)

df['obs_value_bayes_selected_signed_log'] = np.select(
    [
        df['bayes_best_source'] == 'bayes_ref_area',
        df['bayes_best_source'] == 'bayes_activity',
        df['bayes_best_source'] == 'bayes_type_entity',
    ],
    [df['bayes_ref_area'], df['bayes_activity'], df['bayes_type_entity']],
    default=df['bayes_global']
)

# --- Ek modelleme değişkenleri ---
# En sık gözlemlenen 5 aktivite kodu için yüksek etki flag'i
df['high_effect_activity_flag'] = df['ACTIVITY'].isin(
    df['ACTIVITY'].value_counts().head(5).index
).astype(int)

# Panel kimliği: ülke + aktivite + entity tipi kombinasyonu
df['panel_id'] = (
    df['REF_AREA'].astype(str) + '__' +
    df['ACTIVITY'].astype(str) + '__' +
    df['TYPE_ENTITY'].astype(str)
)

display(Markdown(f'**Model karşılaştırmasında kullanılan satır sayısı:** {df.shape[0]:,}'))


## Regresyon Adayları

Üç aday model kademeli olarak değişken ekleyerek karşılaştırılır:
yalnızca anlamlı değişkenler, eksik değer flag'i eklenerek ve
yüksek etki aktivite flag'i eklenerek.


In [ ]:
reg_df = df[[
    'obs_value_bayes_selected_signed_log', 'TYPE_ENTITY', 'MEASURE_PRINCIPLE',
    'TIME_PERIOD', 'obs_missing_flag', 'high_effect_activity_flag'
]].dropna().copy()

# Model 1: yalnızca hipotez testlerinde anlamlı çıkan değişkenler
reg_formula_1 = (
    'obs_value_bayes_selected_signed_log ~ '
    'C(TYPE_ENTITY) + C(MEASURE_PRINCIPLE) + TIME_PERIOD'
)
# Model 2: eksik değer flag'i eklendi — eksikliğin yapısal bilgi taşıyıp taşımadığı test ediliyor
reg_formula_2 = (
    'obs_value_bayes_selected_signed_log ~ '
    'C(TYPE_ENTITY) + C(MEASURE_PRINCIPLE) + TIME_PERIOD + obs_missing_flag'
)
# Model 3: yüksek etki aktivite flag'i eklendi — hipotez testinde anlamlı olmasa da
# etki büyüklüğü karşılaştırmasını hak edecek kadar güçlü olabilir
reg_formula_3 = (
    'obs_value_bayes_selected_signed_log ~ '
    'C(TYPE_ENTITY) + C(MEASURE_PRINCIPLE) + TIME_PERIOD + obs_missing_flag + high_effect_activity_flag'
)

reg_models = {
    'Regression_1_significant_only':        smf.ols(reg_formula_1, data=reg_df).fit(),
    'Regression_2_plus_missing':            smf.ols(reg_formula_2, data=reg_df).fit(),
    'Regression_3_plus_high_effect_nonsig': smf.ols(reg_formula_3, data=reg_df).fit(),
}

reg_compare = pd.DataFrame([
    {'model': name, 'aic': model.aic, 'bic': model.bic, 'rsquared': model.rsquared}
    for name, model in reg_models.items()
]).sort_values(['aic', 'bic'])
reg_compare


## Panel Adayları

Veri seti yalnızca 2023 ve 2024'ü kapsadığından zaman yapısı tam bir
zaman serisi olarak değil, kısa panel olarak ele alınmalıdır.
Pooled OLS, Fixed Effects ve Random Intercept modelleri karşılaştırılıyor.


In [ ]:
panel_df = df.copy().sort_values(['panel_id', 'TIME_PERIOD'])

# İki dönem için anlamlı dinamik değişkenler: lag ve fark
panel_df['lag_bayes_signed_log']  = panel_df.groupby('panel_id')['obs_value_bayes_selected_signed_log'].shift(1)
panel_df['diff_bayes_signed_log'] = panel_df.groupby('panel_id')['obs_value_bayes_selected_signed_log'].diff()

# Pooled OLS: panel yapısını görmezden gelen temel model
pooled = smf.ols(
    'obs_value_bayes_selected_signed_log ~ C(TYPE_ENTITY) + C(MEASURE_PRINCIPLE) + TIME_PERIOD + obs_missing_flag',
    data=panel_df
).fit()

# Fixed Effects: panel_id dummy'leri ile birim içi varyasyonu izole eder
# Not: iki dönemli kısa panelde FE, within-unit variation'u neredeyse
# tamamen açıkladığından AIC üstünlüğü mekanik olabilir
fixed = smf.ols(
    'obs_value_bayes_selected_signed_log ~ C(TIME_PERIOD) + C(panel_id)',
    data=panel_df
).fit()

# Random Intercept: birim etkilerini rastgele olarak modeller
mixed = smf.mixedlm(
    'obs_value_bayes_selected_signed_log ~ C(TYPE_ENTITY) + C(MEASURE_PRINCIPLE) + TIME_PERIOD',
    data=panel_df,
    groups=panel_df['panel_id']
).fit(reml=False, method='lbfgs')

panel_compare = pd.DataFrame([
    {'model': 'Panel_Pooled_OLS',       'aic': pooled.aic, 'bic': pooled.bic},
    {'model': 'Panel_Fixed_Effects',    'aic': fixed.aic,  'bic': fixed.bic},
    {'model': 'Panel_Random_Intercept', 'aic': mixed.aic,  'bic': mixed.bic},
]).sort_values(['aic', 'bic'])
panel_compare


## Zaman Yapılı Adaylar

Yalnızca iki yıllık veriyle klasik uzun vadeli zaman serisi modeli
savunulamaz. Bu bölümdeki modeller yalnızca açıklayıcı bir kıyaslama
noktası olarak sunulmaktadır; birincil çıkarımsal model olarak kullanılmaz.


In [ ]:
# Yıllık ortalama üzerinden iki basit model: sabit seviye ve doğrusal trend
ts_df = (
    df.groupby('TIME_PERIOD', as_index=False)['obs_value_bayes_selected_signed_log']
    .mean()
    .rename(columns={'obs_value_bayes_selected_signed_log': 'annual_mean_bayes'})
)
ts_df['time_index'] = np.arange(len(ts_df))

ts_model_1 = smf.ols('annual_mean_bayes ~ 1',          data=ts_df).fit()  # sabit seviye
ts_model_2 = smf.ols('annual_mean_bayes ~ time_index', data=ts_df).fit()  # doğrusal trend

ts_compare = pd.DataFrame([
    {'model': 'Time_Benchmark_Level', 'aic': ts_model_1.aic, 'bic': ts_model_1.bic},
    {'model': 'Time_Benchmark_Trend', 'aic': ts_model_2.aic, 'bic': ts_model_2.bic},
]).sort_values(['aic', 'bic'])
ts_compare


## AIC/BIC'e Göre Tercih Edilen Modeller


In [ ]:
preferred_reg   = reg_compare.iloc[0]['model']
preferred_panel = panel_compare.iloc[0]['model']
preferred_time  = ts_compare.iloc[0]['model']

comment = (
    f'AIC/BIC\'e göre tercih edilen regresyon modeli: {preferred_reg}. '
    f'AIC/BIC\'e göre tercih edilen panel modeli: {preferred_panel}. '
    f'En iyi zaman yapılı kıyaslama noktası {preferred_time} olmakla birlikte, '
    'veri seti yalnızca iki zaman noktası içerdiğinden bu model birincil '
    'çıkarımsal model olarak ele alınmaz. Bu nedenle nihai proje yorumunda '
    'panel ailesi zaman serisi ailesine tercih edilmelidir.'
)
display(Markdown('### Yorum\n\n' + comment))
